# Attention MNIST

This is a simple notebook for me to help get the basics of softmax-attention down to a tea. This implementation will be from memory.

Given an input sequence of embedded-tokens $N$ tokens $X \in \mathbb{R}^{N \times d_\text{emb}}$

Scaled Dot-product Attention is formulated as 
$$\text{Softmax}\left(\frac{QK^\top}{\sqrt{d_\text{hidden}}}\right)V$$

Where some matriices $W_Q, W_K \in \mathbb{R}^{d_\text{emb}\times d_\text{hidden}}$ and $W_V \in \mathbb{R}^{d_\text{emb} \times d_\text{emb}}$ and

$$Q = XW_Q,\quad  K = XW_K, \quad V= XW_V$$

Though it is not often expressed as such, we can write attention as a function 
$$\text{Attention}: \mathbb{R}^{N \times d_\text{emb}} \rightarrow \mathbb{R}^{N \times d_\text{emb}}$$

with form
$$\text{Attention}(X; W_Q, W_K, Q_V) = \text{RowSoftmax}\left(\frac{XW_QW_K^\top X^\top}{\sqrt{d_k}}\right)XW_V$$




In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SoftmaxAttention(nn.Module):
    def __init__(self, d_emb, d_hidden):
        super().__init__()
        self.W_Q = nn.Linear(d_emb, d_hidden, bias=False)
        self.W_K = nn.Linear(d_emb, d_hidden, bias=False)
        self.W_V = nn.Linear(d_emb, d_emb, bias=False)

    def forward(self, X):
        """
        X: (N, d_emb), where N is the number of tokens in the sequence
        Returns: (N, d_emb), same shape as input
        """
        Q = self.W_Q(X)               # (N, d_hidden)
        K = self.W_K(X)               # (N, d_hidden)
        V = self.W_V(X)               # (N, d_emb)

        d_k = Q.size(-1)
        attention_scores = torch.matmul(Q, K.t()) / math.sqrt(d_k)   # (N, N)
        attention_weights = F.softmax(attention_scores, dim=-1)      # (N, N)
        output = torch.matmul(attention_weights, V)                  # (N, d_emb)
        return output

d_emb = 64
d_hidden = 32
N = 50

attention = SoftmaxAttention(d_emb, d_hidden)
X = torch.randn(N, d_emb)
out = attention(X)

print("X shape:", X.shape)
print("Output shape:", out.shape)

X shape: torch.Size([50, 64])
Output shape: torch.Size([50, 64])


### Position Embeddings

It is important to realize that there is no explicit notion of position in the attention architecture...

If I had some $N$-sequence $d_\text{emb}$ embeddings encoded in matrix $X \in \mathbb{R}^{N \times d_\text{emb}}$... Then I could permute the rows of the matrix to get $\hat X$, feed this permuted sequence into my attention and get the same output after unpermuting the output. 

You can prove this by defining $\hat X = XP$ for some permutation matrix $P$ and showing $$\text{Attention(X)} = \text{Attention(XP)}P^{-1}$$ (Exercise for reader)

However, all implementations of Attention use a trick called "positional-embeddings" to make the position of the $n$-th element of the sequence explicit in the embedding itself.

In the *simplest* form, we can take a sequence of *constant* embeddings $e_1, e_2, ..., e_N \in \mathbb{R}^{d_\text{emb}}$ to form an embedding matrix $E$ and add them to our original tokens:

$$X_\text{new} = X + E$$

Notably, $$XP + E \neq (X+E)P$$ 
so if we pass $X_\text{new}$ to $\text{Attention}(\cdot)$, the function is no longer permutation invariant. There is a lot of research that goes into what $e_t$ should be. One of the simplest things to do is to set them to random vectors and let them be learnable parameters.

In [29]:
def add_positional_embeddings(X, pos_embedding):
    N = X.size(0)
    positions = torch.arange(N, device=X.device) # [0, 1, 2, ... N-1]
    E = pos_embedding(positions)  # (N, d_emb)
    return X + E

d_emb = 32
num_tokens = 50
embedding_matrix = nn.Embedding(num_tokens, d_emb)

### Running our Transformer MNIST

MNIST as you might be familiar with is one of the simplest deep learning tasks. It consists of 28x28 pixel images handwritten digits and labels for what digit is printed.

In order to plug these into our attention mechanism we have two challenges. 
 1) How do we turn the image into a sequence of embedded tokens
 2) How do we turn the output sequence into a class prediction

**Patchification**

This method addresses 1) from above. We can break the 28x28 image into a grid of 4x4 non-overlapping patches. There would be $7 \times 7=49$ patches in this image. (GET VISUAL). Then each of these $4\times 4=16$ dimensional patches can be encoded into $d_\text{emb}$ before being given positions and passed through the attention mechanism. The patch encoder can be a linear projection by parameters $W_X$ \in \mathbb{R}^{4 \times {d_\text{emb}}}$

**Class Token**

To the end of our sequence of $49$ patches we can add a dummy "Class" token often denoted as [CLS] in the literature. After passing 50 token sequence (patches + [CLS]) through the attention mechanism we recieve final embedding of the [CLS].

This final [CLS] embedding can be projected into a softmax logits over the 10 digit labels, of which the argmax will tell us the class. 


In [30]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

# Download MNIST dataset
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

def patchify_and_append_class_token(images, patch_proj, patch_size=4):
    B, C, H, W = images.shape
    assert H % patch_size == 0 and W % patch_size == 0, "Image size must be divisible by patch size."
    assert patch_proj.in_features == C * patch_size * patch_size, f"patch_proj.in_features ({patch_proj.in_features}) does not match number of patch elements ({C * patch_size * patch_size})"

    num_patches_h = H // patch_size
    num_patches_w = W // patch_size
    num_patches = num_patches_h * num_patches_w

    # Create patches: (B, C, num_patches_h, patch_size, num_patches_w, patch_size)
    patches = images.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
    # Rearrange to (B, num_patches, patch_dim)
    patches = patches.permute(0,2,4,1,3,5).contiguous()
    patches = patches.view(B, num_patches, C * patch_size * patch_size)

    # Project each patch to d_emb
    patch_embeddings = patch_proj(patches)  # (B, num_patches, d_emb)
    d_emb = patch_embeddings.shape[-1]

    # Prepare class token - zeros
    cls_token = torch.zeros(B, 1, d_emb, device=images.device)
    patch_embeddings_with_cls = torch.cat([patch_embeddings, cls_token], dim=1) # (B, num_patches+1, d_emb)
    return patch_embeddings_with_cls


# Sample a batch from the data loader
batch = next(iter(train_loader))
images, labels = batch
print("Batch images shape:", images.shape)
print("Batch labels shape:", labels.shape)

# Create a dummy patch projection layer
C, patch_size = 1, 4
d_emb = 32
patch_dim = C * patch_size * patch_size
W_X = nn.Linear(patch_dim, d_emb)

# Pass the images through patchify_and_append_class_token
patchified = patchify_and_append_class_token(images, W_X, patch_size=patch_size)
print("Patchified + appended [CLS] token shape:", patchified.shape)

Batch images shape: torch.Size([64, 1, 28, 28])
Batch labels shape: torch.Size([64])
Patchified + appended [CLS] token shape: torch.Size([64, 50, 32])


### Putting it all together

We are about ready to train, but before we do, let's enumerate what our trainable parameters are:

1) Attention weights: $W_Q, W_K, W_V$
2) Positional Embeddings: $E$
3) Patch Encoder: $W_X$
4) Prediction Head: $W_L$ 

We already instantiaed these objects in the cells that introduced them so we just need to reuse them for training.

In [31]:
from tqdm import tqdm

# Define positional embeddings, attention, and a prediction head
num_patches = (28 // patch_size) * (28 // patch_size)
E = nn.Embedding(num_patches + 1, d_emb).to(images.device)  # +1 for CLS
attention = nn.MultiheadAttention(d_emb, num_heads=2, batch_first=True).to(images.device)
pred_head = nn.Linear(d_emb, 10).to(images.device)

params = list(W_X.parameters()) + list(E.parameters()) + list(attention.parameters()) + list(pred_head.parameters())
optimizer = torch.optim.Adam(params, lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# Helper function to process a batch of images through patchify_and_append_class_token all the way to logits
def forward_transformer(images):
    """
    images: (B, C, H, W) torch tensor
    Returns: logits (B, 10)
    """
    B = images.shape[0]
    device = images.device
    X_seq = patchify_and_append_class_token(images, W_X, patch_size=patch_size)
    pos_ids = torch.arange(X_seq.shape[1], device=device).unsqueeze(0).repeat(B,1)  # (B, num_patches+1)
    X_seq = X_seq + E(pos_ids)
    attn_out, _ = attention(X_seq, X_seq, X_seq)
    logits = pred_head(attn_out[:, -1, :])  # Use CLS token output
    return logits

epochs = 3
for epoch in range(epochs):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for images, y in pbar:
        images, y = images.to(images.device), y.to(images.device)
        logits = forward_transformer(images)
        loss = loss_fn(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pbar.set_postfix({"loss": loss.item()})
    print(f"Epoch {epoch+1} - Last batch loss: {loss.item()}")

    # Validation loop on the test data
    attention.eval()
    pred_head.eval()
    E.eval()
    W_X.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, y in test_loader:
            images, y = images.to(images.device), y.to(images.device)
            logits = forward_transformer(images)
            val_loss += loss_fn(logits, y).item() * images.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == y).sum().item()
            val_total += images.size(0)
    avg_val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    print(f"Validation loss: {avg_val_loss:.4f}, Validation accuracy: {val_acc:.4f}")
    attention.train()
    pred_head.train()
    E.train()
    W_X.train()

Epoch 1: 100%|██████████| 938/938 [00:23<00:00, 40.72it/s, loss=0.363]


Epoch 1 - Last batch loss: 0.36258968710899353
Validation loss: 0.7346, Validation accuracy: 0.7636


Epoch 2: 100%|██████████| 938/938 [00:23<00:00, 40.75it/s, loss=0.658]


Epoch 2 - Last batch loss: 0.6576557159423828
Validation loss: 0.5546, Validation accuracy: 0.8311


Epoch 3: 100%|██████████| 938/938 [00:23<00:00, 40.73it/s, loss=0.601]


Epoch 3 - Last batch loss: 0.6014688014984131
Validation loss: 0.4759, Validation accuracy: 0.8562


# Interpretability

Hooray! The above is as minimal of a "Transformer" implementation as you might get. Now that we have it with all it's internals opened up, there are a lot of things we can do to understand what is going on.

I think the first thing to look at is the "Attention Matrix" 
$$A = \text{Softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)$$

As a reminder the logits can be expressed as follows.

$$\text{logits}(A, V, W_S) = \left[AV\right]_{-1}W_S = \left[A\right]_{-1} VW_S$$


Where $[\cdot]_{-1}$ indicates grabbing the last row (i.e. the row corresponding to the [CLS] token).




**Some (extra) comments:**
The pre-softmax attention matrix $A = \frac{XW_QW_K^\top X^\top}{\sqrt{d_k}}$ can be thought of as a the pair-wise weigthed inner products of all sequence elements. In other words

$$A_{ij} = \left\langle X_i, X_j \right\rangle_{W_QW_K^\top}$$

